# 6. Advanced RAG


In [1]:
!pip install numpy==1.26.4

In [2]:
# 【注意】
# 上記の `!pip install numpy==1.26.4` を実行したあと、
# Google Colab 上部のメニューから「ランタイム」の「セッションを再起動する」を実行してください。
# その後このセルを実行して `1.26.4` と表示されることを確認してください。

import numpy as np

print(np.__version__)
assert np.__version__ == "1.26.4"

1.26.4


In [2]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = "agent-book"
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

## 6.2. ハンズオンの準備


In [4]:
!pip install langchain-core==0.3.0 langchain-openai==0.2.0 \
    langchain-community==0.3.0 GitPython==3.1.43 \
    langchain-chroma==0.1.4 tavily-python==0.5.0 pydantic==2.10.6

In [3]:
from langchain_community.document_loaders import GitLoader


def file_filter(file_path: str) -> bool:
    return file_path.endswith(".mdx")


loader = GitLoader(
    clone_url="https://github.com/langchain-ai/langchain",
    repo_path="./langchain",
    branch="langchain==0.2.13",
    file_filter=file_filter,
)

documents = loader.load()
print(len(documents))

280


In [4]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
db = Chroma.from_documents(documents, embeddings)

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [5]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_template('''\
以下の文脈だけを踏まえて質問に回答してください。

文脈: """
{context}
"""

質問: {question}
''')

model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

retriever = db.as_retriever()

chain = {
    "question": RunnablePassthrough(),
    "context": retriever,
} | prompt | model | StrOutputParser()

chain.invoke("LangChainの概要を教えて")

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


'LangChainは、大規模言語モデル（LLM）を活用したアプリケーションを開発するためのフレームワークです。このフレームワークは、LLMアプリケーションのライフサイクルの各段階を簡素化します。具体的には、以下のような機能を提供しています。\n\n1. **開発**: LangChainのオープンソースのビルディングブロックやコンポーネント、サードパーティの統合を使用してアプリケーションを構築できます。また、LangGraphを利用することで、状態を持つエージェントを構築し、ストリーミングや人間の介入をサポートします。\n\n2. **生産化**: LangSmithを使用して、チェーンを検査、監視、評価し、継続的に最適化して自信を持ってデプロイできます。\n\n3. **デプロイ**: LangGraphアプリケーションを生産準備が整ったAPIやアシスタントに変換できます。\n\nLangChainは、以下のオープンソースライブラリで構成されています：\n- `langchain-core`: 基本的な抽象化とLangChain表現言語（LCEL）。\n- `langchain-community`: サードパーティの統合。\n- `langchain`: アプリケーションの認知アーキテクチャを構成するチェーン、エージェント、検索戦略。\n- LangGraph: LLMを用いた堅牢で状態を持つマルチアクターアプリケーションを構築するためのツール。\n\nLangChainは、Pythonライブラリに焦点を当てており、JavaScript版のドキュメントも提供されています。'

## 6.3. 検索クエリの工夫


### HyDE（Hypothetical Document Embeddings）


In [9]:
hypothetical_prompt = ChatPromptTemplate.from_template("""\
次の質問に回答する一文を書いてください。

質問: {question}
""")

hypothetical_chain = hypothetical_prompt | model | StrOutputParser()

In [10]:
hyde_rag_chain = {
    "question": RunnablePassthrough(),
    "context": hypothetical_chain | retriever,
} | prompt | model | StrOutputParser()

hyde_rag_chain.invoke("LangChainの概要を教えて")

'LangChainは、LLM（大規模言語モデル）を活用したアプリケーションの開発を支援するフレームワークです。主に、以下のような機能や特徴を持っています。\n\n1. **統一インターフェース**: LangChainは、LCEL（LangChain Expression Language）を使用して、さまざまなコンポーネントを統一的に扱うことができます。これにより、ストリーミングやバッチ処理などの操作が一貫してサポートされます。\n\n2. **コンポジションプリミティブ**: 複雑なチェーンを簡単に構成したり、並列処理を行ったりするためのプリミティブが提供されています。\n\n3. **LangGraph**: LCELの上に構築されたLangGraphは、アプリケーションコンポーネントの効率的なオーケストレーションを可能にし、可読性の高いコードを維持します。\n\n4. **多様なモデルのサポート**: LangChainは、さまざまなプロバイダーからのモデル（IBM、Clarifaiなど）を統合し、LLM、埋め込みモデル、ベクトルストアなどを利用できます。\n\n5. **チュートリアルとドキュメント**: 新しいユーザー向けに、基本的なアプリケーションの構築から、特定のタスクに特化したチュートリアルまで、豊富なリソースが提供されています。\n\nLangChainは、AIアプリケーションの開発を迅速かつ効率的に行うための強力なツールセットを提供しています。'

### 複数の検索クエリの生成


In [11]:
from pydantic import BaseModel, Field


class QueryGenerationOutput(BaseModel):
    queries: list[str] = Field(..., description="検索クエリのリスト")


query_generation_prompt = ChatPromptTemplate.from_template("""\
質問に対してベクターデータベースから関連文書を検索するために、
3つの異なる検索クエリを生成してください。
距離ベースの類似性検索の限界を克服するために、
ユーザーの質問に対して複数の視点を提供することが目標です。

質問: {question}
""")

query_generation_chain = (
    query_generation_prompt
    | model.with_structured_output(QueryGenerationOutput)
    | (lambda x: x.queries)
)

In [12]:
multi_query_rag_chain = {
    "question": RunnablePassthrough(),
    "context": query_generation_chain | retriever.map(),
} | prompt | model | StrOutputParser()

multi_query_rag_chain.invoke("LangChainの概要を教えて")

'LangChainは、大規模言語モデル（LLM）を活用したアプリケーションを開発するためのフレームワークです。このフレームワークは、LLMアプリケーションのライフサイクルの各ステージを簡素化します。具体的には、以下のような機能を提供しています。\n\n1. **開発**: LangChainのオープンソースのビルディングブロック、コンポーネント、サードパーティの統合を使用してアプリケーションを構築できます。また、LangGraphを使用して、状態を持つエージェントを構築し、ストリーミングや人間の介入をサポートします。\n\n2. **生産化**: LangSmithを使用して、チェーンを検査、監視、評価し、継続的に最適化して自信を持ってデプロイできます。\n\n3. **デプロイ**: LangGraphアプリケーションを生産準備が整ったAPIやアシスタントに変換することができます。\n\nLangChainは、以下のオープンソースライブラリで構成されています：\n- **`langchain-core`**: 基本的な抽象化とLangChain表現言語。\n- **`langchain-community`**: サードパーティの統合。\n- **`langchain`**: アプリケーションの認知アーキテクチャを構成するチェーン、エージェント、検索戦略。\n- **LangGraph**: LLMを使用して堅牢で状態を持つマルチアクターアプリケーションを構築するためのライブラリ。\n- **LangServe**: LangChainのチェーンをREST APIとしてデプロイするためのツール。\n- **LangSmith**: LLMアプリケーションをデバッグ、テスト、評価、監視するための開発者プラットフォーム。\n\nLangChainは、PythonとJavaScriptの両方のライブラリに対応しており、さまざまな統合やチュートリアルも提供されています。'

## 6.4. 検索後の工夫


### RAG Fusion


In [19]:
from langchain_core.documents import Document


def reciprocal_rank_fusion(
    retriever_outputs: list[list[Document]],
    k: int = 60,
) -> list[str]:
    # 各ドキュメントのコンテンツ (文字列) とそのスコアの対応を保持する辞書を準備
    content_score_mapping = {}

    # 検索クエリごとにループ
    for docs in retriever_outputs:
        # 検索結果のドキュメントごとにループ
        for rank, doc in enumerate(docs, start=1):
            content = doc.page_content

            # 初めて登場したコンテンツの場合はスコアを0で初期化
            if content not in content_score_mapping:
                content_score_mapping[content] = 0

            # (1 / (順位 + k)) のスコアを加算
            content_score_mapping[content] += 1 / (rank + k)

    # スコアの大きい順にソート
    ranked = sorted(content_score_mapping.items(), key=lambda x: x[1], reverse=True)  # noqa
    return [content for content, _ in ranked]

In [14]:
rag_fusion_chain = {
    "question": RunnablePassthrough(),
    "context": query_generation_chain | retriever.map() | reciprocal_rank_fusion,
} | prompt | model | StrOutputParser()

rag_fusion_chain.invoke("LangChainの概要を教えて")

'LangChainは、大規模言語モデル（LLMs）を活用したアプリケーションを開発するためのフレームワークです。このフレームワークは、LLMアプリケーションのライフサイクルの各段階を簡素化します。具体的には、以下のような機能があります。\n\n1. **開発**: LangChainのオープンソースのビルディングブロックやコンポーネント、サードパーティの統合を使用してアプリケーションを構築できます。また、LangGraphを使用して、状態を持つエージェントを構築し、ストリーミングや人間の介入をサポートします。\n\n2. **生産化**: LangSmithを使用して、チェーンを検査、監視、評価し、継続的に最適化して自信を持ってデプロイできます。\n\n3. **デプロイ**: LangGraphアプリケーションを本番環境向けのAPIやアシスタントに変換することができます。\n\nLangChainは、以下のオープンソースライブラリで構成されています：\n- **`langchain-core`**: 基本的な抽象化とLangChain表現言語。\n- **`langchain-community`**: サードパーティの統合。\n- **`langchain`**: アプリケーションの認知アーキテクチャを構成するチェーン、エージェント、検索戦略。\n- **LangGraph**: LLMを使用して堅牢で状態を持つマルチアクターアプリケーションを構築するためのライブラリ。\n- **LangServe**: LangChainチェーンをREST APIとしてデプロイするためのツール。\n- **LangSmith**: LLMアプリケーションをデバッグ、テスト、評価、監視するための開発者プラットフォーム。\n\nLangChainは、PythonとJavaScriptの両方のライブラリがあり、特にPythonのドキュメントに焦点を当てています。'

### Cohere のリランクモデルを使用する準備


In [6]:
os.environ["COHERE_API_KEY"] = userdata.get("COHERE_API_KEY")

In [7]:
!pip install langchain-cohere==0.3.0

### Cohere のリランクモデルの導入


In [8]:
from typing import Any

from langchain_cohere import CohereRerank
from langchain_core.documents import Document


def rerank(inp: dict[str, Any], top_n: int = 3) -> list[Document]:
    question = inp["question"]
    documents = inp["documents"]

    cohere_reranker = CohereRerank(model="rerank-multilingual-v3.0", top_n=top_n)
    return cohere_reranker.compress_documents(documents=documents, query=question)


rerank_rag_chain = (
    {
        "question": RunnablePassthrough(),
        "documents": retriever,
    }
    | RunnablePassthrough.assign(context=rerank)
    | prompt | model | StrOutputParser()
)

rerank_rag_chain.invoke("LangChainの概要を教えて")

'LangChainは、大規模言語モデル（LLM）を活用したアプリケーションを開発するためのフレームワークです。このフレームワークは、LLMアプリケーションのライフサイクルの各段階を簡素化します。具体的には、以下のような機能を提供しています。\n\n- **開発**: LangChainのオープンソースのビルディングブロックやコンポーネント、サードパーティの統合を使用してアプリケーションを構築できます。また、LangGraphを使用して、状態を持つエージェントを構築し、ストリーミングや人間の介入をサポートします。\n\n- **生産化**: LangSmithを利用して、チェーンを検査、監視、評価し、継続的に最適化して自信を持ってデプロイできます。\n\n- **デプロイ**: LangGraphアプリケーションを生産準備が整ったAPIやアシスタントに変換することができます。\n\nLangChainは、以下のオープンソースライブラリで構成されています：\n- `langchain-core`: 基本的な抽象化とLangChain表現言語。\n- `langchain-community`: サードパーティの統合。\n- `langchain`: アプリケーションの認知アーキテクチャを構成するチェーン、エージェント、検索戦略。\n- LangGraph: LLMを使用して堅牢で状態を持つマルチアクターアプリケーションを構築するためのツール。\n- LangServe: LangChainチェーンをREST APIとしてデプロイするためのツール。\n- LangSmith: LLMアプリケーションをデバッグ、テスト、評価、監視するための開発者プラットフォーム。\n\nLangChainは、PythonとJavaScriptの両方のライブラリに対応しており、さまざまなチュートリアルやガイドが用意されています。'

## 6.5. 複数の Retriever を使う工夫


### LLM によるルーティング


In [9]:
from langchain_community.retrievers import TavilySearchAPIRetriever

langchain_document_retriever = retriever.with_config(
    {"run_name": "langchain_document_retriever"}
)

web_retriever = TavilySearchAPIRetriever(k=3).with_config(
    {"run_name": "web_retriever"}
)

In [12]:
from enum import Enum


class Route(str, Enum):
    langchain_document = "langchain_document"
    web = "web"


class RouteOutput(BaseModel):
    route: Route


route_prompt = ChatPromptTemplate.from_template("""\
質問に回答するために適切なRetrieverを選択してください。

質問: {question}
""")

route_chain = (
    route_prompt
    | model.with_structured_output(RouteOutput)
    | (lambda x: x.route)
)

In [13]:
def routed_retriever(inp: dict[str, Any]) -> list[Document]:
    question = inp["question"]
    route = inp["route"]

    if route == Route.langchain_document:
        return langchain_document_retriever.invoke(question)
    elif route == Route.web:
        return web_retriever.invoke(question)

    raise ValueError(f"Unknown route: {route}")


route_rag_chain = (
    {
        "question": RunnablePassthrough(),
        "route": route_chain,
    }
    | RunnablePassthrough.assign(context=routed_retriever)
    | prompt | model | StrOutputParser()
)

In [14]:
route_rag_chain.invoke("LangChainの概要を教えて")

'LangChainは、大規模言語モデル（LLM）を活用したアプリケーションを開発するためのフレームワークです。このフレームワークは、LLMアプリケーションのライフサイクルの各段階を簡素化します。具体的には、以下のような機能を提供しています。\n\n1. **開発**: LangChainのオープンソースのビルディングブロックやコンポーネント、サードパーティの統合を使用してアプリケーションを構築できます。また、LangGraphを利用することで、状態を持つエージェントを構築し、ストリーミングや人間の介入をサポートします。\n\n2. **生産化**: LangSmithを使用して、チェーンを検査、監視、評価し、継続的に最適化して自信を持ってデプロイできます。\n\n3. **デプロイ**: LangGraphアプリケーションを生産準備が整ったAPIやアシスタントに変換できます。\n\nLangChainは、以下のオープンソースライブラリで構成されています：\n- `langchain-core`: 基本的な抽象化とLangChain表現言語（LCEL）。\n- `langchain-community`: サードパーティの統合。\n- `langchain`: アプリケーションの認知アーキテクチャを構成するチェーン、エージェント、検索戦略。\n- LangGraph: LLMを用いた堅牢で状態を持つマルチアクターアプリケーションを構築するためのツール。\n\nLangChainは、Pythonライブラリに焦点を当てており、JavaScript版のドキュメントも提供されています。'

In [15]:
route_rag_chain.invoke("東京の今日の天気は？")

'東京の今日の天気は、曇りがちで、時折晴れ間も見られる予報です。日中の気温は28度前後まで上がる見込みで、湿度も高めで蒸し暑く感じられるでしょう。'

### ハイブリッド検索の実装


In [16]:
!pip install rank-bm25==0.2.2

In [17]:
from langchain_community.retrievers import BM25Retriever

chroma_retriever = retriever.with_config(
    {"run_name": "chroma_retriever"}
)

bm25_retriever = BM25Retriever.from_documents(documents).with_config(
    {"run_name": "bm25_retriever"}
)

In [20]:
from langchain_core.runnables import RunnableParallel

hybrid_retriever = (
    RunnableParallel({
        "chroma_documents": chroma_retriever,
        "bm25_documents": bm25_retriever,
    })
    | (lambda x: [x["chroma_documents"], x["bm25_documents"]])
    | reciprocal_rank_fusion
)

In [21]:
hybrid_rag_chain = (
    {
        "question": RunnablePassthrough(),
        "context": hybrid_retriever,
    }
    | prompt | model | StrOutputParser()
)

hybrid_rag_chain.invoke("LangChainの概要を教えて")

'LangChainは、大規模言語モデル（LLM）を活用したアプリケーションを開発するためのフレームワークです。このフレームワークは、LLMアプリケーションのライフサイクルの各段階を簡素化します。具体的には、以下のような機能を提供しています。\n\n- **開発**: LangChainのオープンソースのビルディングブロックやコンポーネント、サードパーティの統合を使用してアプリケーションを構築できます。また、LangGraphを利用することで、状態を持つエージェントを構築し、ストリーミングや人間の介入をサポートします。\n\n- **生産化**: LangSmithを使用して、チェーンを検査、監視、評価し、継続的に最適化して自信を持ってデプロイできます。\n\n- **デプロイ**: LangGraphアプリケーションを生産準備が整ったAPIやアシスタントに変換できます。\n\nLangChainは、以下のオープンソースライブラリで構成されています：\n- `langchain-core`: 基本的な抽象化とLangChain表現言語（LCEL）。\n- `langchain-community`: サードパーティの統合。\n- `langchain`: アプリケーションの認知アーキテクチャを構成するチェーン、エージェント、検索戦略。\n- LangGraph: LLMを用いた堅牢で状態を持つマルチアクターアプリケーションを構築するためのツール。\n\nこのように、LangChainはLLMアプリケーションの開発を効率化し、さまざまな機能を提供するフレームワークです。'